# Recommender: inspect each stage

Run cells from top to bottom. HF lookup, Tavily search, and Nemotron validation are separate calls that share saved evidence. The notebook uses the existing Linux `.venv` through WSL when the Jupyter kernel is on Windows. It does not install packages or launch a benchmark. Keep API keys in the repository `.env`, not in this notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, sys

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'config/recommender.yaml').exists()), None)
assert repo_root is not None, 'Open the notebook from the repository or notebooks/ folder'
linux_python = repo_root / '.venv/bin/python'
windows_python = repo_root / '.venv/Scripts/python.exe'
if os.name == 'nt' and linux_python.exists():
    assert shutil.which('wsl.exe'), 'WSL is needed to run the Linux .venv'
    assert repo_root.drive, 'Use a WSL kernel for a repository on a UNC path'
    wsl_root = f'/mnt/{repo_root.drive[0].lower()}/{repo_root.relative_to(repo_root.anchor).as_posix()}'
    python_runner = f'wsl.exe --cd "{wsl_root}" -- ./.venv/bin/python'
elif os.name == 'nt' and windows_python.exists():
    python_runner = r'.\.venv\Scripts\python.exe'
else:
    # Use this notebook's selected WSL kernel, not the project's GPU .venv.
    python_runner = f'env PYTHONPATH=src {sys.executable}'
model_id = 'Qwen/Qwen3.5-4B'
task_request = 'Find a good model with a speculative decoding head for one 80GB A100 GPU, a budget of $5/hour, and a maximum of 2000 generated tokens; any license, any framework, sweep mode.'
demo_dir = Path('artifacts/recommendations') / f"notebook-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
demo_dir_cli = demo_dir.as_posix()
print('Model:', model_id, '\nOutput:', repo_root / demo_dir)

In [ ]:
%cd {repo_root}

## 1. Check keys

This checks the **WSL process** and repository `.env` without printing secrets. Nemotron needs `NEBIUS_API_KEY`; Tavily needs `TAVILY_API_KEY`. If Tavily says `false`, add the key to `.env` and rerun this cell.

In [ ]:
!{python_runner} -m hf_model_watch.recommender.probe status --output-dir {demo_dir_cli}

## 2. Hugging Face scan

Look up one exact model. The cell shows the revision, license, parameters, context, architecture, chat-template presence, size, missing fields, and card path. The complete scan record is saved as `hf_record.json`. If this cell fails, stop here: later cells need its artifact.

In [ ]:
!{python_runner} -m hf_model_watch.recommender.probe hf --model-id {model_id} --output-dir {demo_dir_cli}

## 3. Tavily research

Run focused searches for fields HF did not establish, plus speculative decoding and runtime evidence. The following cell displays every saved query and result, including source text and URLs. The data is saved in `tavily_searches.json` and `tavily_evidence.json`. A missing key produces an empty evidence file.

In [ ]:
!{python_runner} -m hf_model_watch.recommender.probe tavily --output-dir {demo_dir_cli} --request "{task_request}"

In [ ]:
from html import escape
from IPython.display import HTML, display

searches_path = repo_root / demo_dir / 'tavily_searches.json'
evidence_path = repo_root / demo_dir / 'tavily_evidence.json'
if not evidence_path.exists():
    raise FileNotFoundError(f'Run the Tavily cell first: {evidence_path}')
searches = json.loads(searches_path.read_text(encoding='utf-8')) if searches_path.exists() else []
sources = json.loads(evidence_path.read_text(encoding='utf-8'))
if not searches:
    display(HTML('<p>No searches ran. Check the Tavily status output above.</p>'))
for number, search in enumerate(searches, 1):
    matches = [row for row in sources if row.get('research_case') == search['case']]
    title = f'{number}. {search["case"]}: {search["query"]}'
    display(HTML(f'<h3>{escape(title)}</h3><p>{len(matches)} saved results' + (f'; error: {escape(search["error"])}' if search.get('error') else '') + '</p>'))
    for index, row in enumerate(matches, 1):
        url = str(row.get('source_url') or '')
        label = escape(str(row.get('title') or url))
        body = escape(str(row.get('retrieved_text') or 'No source text returned.'))
        display(HTML(f'<details><summary>{index}. <a href="{escape(url, quote=True)}" target="_blank">{label}</a> (score: {escape(str(row.get("score")))})</summary><pre style="white-space:pre-wrap;overflow-wrap:anywhere">{body}</pre></details>'))


## 4. Nemotron validation

This call uses the **saved** HF and Tavily artifacts. The next cell displays the full Nemotron reasoning, final JSON, cited URLs, and the final recommendation. This example uses the stated $5/hour A100 price assumption; replace it with your actual rental price before deployment. A documented drafter pair can pass 80 GB capacity while A100 speedup remains unmeasured.

In [ ]:
!{python_runner} -m hf_model_watch.recommender.probe nemotron --output-dir {demo_dir_cli} --request "{task_request}" --gpu-memory-gb 80 --budget-usd-per-hour 5 --gpu-hourly-cost-usd 5 --license any --framework-mode sweep

In [ ]:
from IPython.display import Markdown, display

response_path = repo_root / demo_dir / 'nemotron_validation_response.json'
if not response_path.exists():
    raise FileNotFoundError(f'Run the Nemotron cell first: {response_path}')
response = json.loads(response_path.read_text(encoding='utf-8'))
message = response.get('message', response)  # supports artifacts from earlier notebook versions
reasoning = message.get('reasoning') or message.get('reasoning_content')
final_answer = message.get('content')

display(Markdown('## Nemotron reasoning'))
display(Markdown(str(reasoning) if reasoning else '*No reasoning returned by the API.*'))
try:
    answer = json.loads(final_answer)
except (TypeError, ValueError) as exc:
    answer = {}
    display(Markdown(f'## Invalid Nemotron answer\nThe API returned text that is not one JSON object: `{exc}`. The complete raw answer is saved in `{response_path}`.'))
    display(HTML(f'<details><summary>Show raw answer</summary><pre style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(str(final_answer or ""))}</pre></details>'))
else:
    display(Markdown('## Final answer'))
    display(Markdown(f'```json\n{json.dumps(answer, indent=2, ensure_ascii=False)}\n```'))
saved_sources = json.loads((repo_root / demo_dir / 'tavily_evidence.json').read_text(encoding='utf-8'))
saved_urls = {row.get('source_url') for row in saved_sources}
field_urls = {url for fields in answer.get('field_assessments', {}).values() for field in fields.values() for url in field.get('sources', [])}
reasoning_urls = {url for url in saved_urls if url and url in str(answer.get('reasoning') or '') + str(reasoning or '')}
citations = sorted(field_urls | reasoning_urls)
display(Markdown('## Cited sources'))
display(HTML(''.join(f'<p><a href="{escape(url, quote=True)}" target="_blank">{escape(url)}</a></p>' for url in citations) or '<p>No saved source URLs cited in the valid answer or API reasoning.</p>'))
recommendation = json.loads((repo_root / demo_dir / 'recommendation.json').read_text(encoding='utf-8'))
display(Markdown('## Recommendation'))
chosen = recommendation.get('recommendation')
if chosen and chosen['speculative_decoding']['usable']:
    pair = chosen['speculative_decoding']
    display(Markdown(f'**{chosen["model_id"]}** with **{pair["draft_model"]}** via **{pair["runtime"]}**. Estimated memory: **{chosen["estimated_bf16_memory_gb"]:.1f} GB** of 80 GB. A100 speedup: **{pair["a100_performance"]}**. [Pair evidence]({pair["source_url"]}).'))
else:
    display(Markdown(f'No eligible recommendation. Blockers: `{recommendation.get("evaluated_candidates", [{}])[0].get("blocking_reasons", [])}`'))
print('Finish reason:', response.get('finish_reason'))